# MapleStory Chatbot Dataset EDA

이 노트북은 정리 후 전달본 기준으로 작성되었습니다. 현재 폴더에는 최종 통합 CSV 1개와 설명 파일만 포함되어 있습니다.


## 현재 파일 구성

- `maple_chatbot_final_dataset.csv`: 최종 챗봇용 통합 데이터셋
- `README_HANDOFF.md`: 인수인계 설명 파일
- `dataset_summary.csv`: 데이터셋 요약
- `mapleqa_eda_for_chatbot.ipynb`: 이 노트북

원본 raw 데이터는 상위 폴더의 `data/raw/`에 보존되어 있습니다.


In [ ]:
import pandas as pd
from pathlib import Path

DATASET_CANDIDATES = [Path('maple_chatbot_final_dataset.csv'), Path('handoff_simplified/maple_chatbot_final_dataset.csv')]
SUMMARY_CANDIDATES = [Path('dataset_summary.csv'), Path('handoff_simplified/dataset_summary.csv')]
DATASET = next((p for p in DATASET_CANDIDATES if p.exists()), None)
SUMMARY = next((p for p in SUMMARY_CANDIDATES if p.exists()), None)
if DATASET is None or SUMMARY is None:
    raise FileNotFoundError('maple_chatbot_final_dataset.csv 또는 dataset_summary.csv를 찾을 수 없습니다.')

df = pd.read_csv(DATASET, encoding='utf-8-sig')
summary = pd.read_csv(SUMMARY, encoding='utf-8-sig')
print(f'dataset={DATASET}')
print(f'rows={len(df):,}, columns={len(df.columns):,}')
display(df.head(3))


## 최종 데이터셋 개요

최종 CSV는 기존 공식/위키/API 샘플 데이터에 보스 추천/장비 성장 추천 룰, 추천 답변 가드레일, 추가 보강 백로그를 같은 스키마로 합친 파일입니다.


In [ ]:
display(summary[summary['section'].eq('file')])
display(summary[summary['section'].eq('collection_scope')])


## 카테고리 분포

`category`는 챗봇 검색/필터링에서 가장 기본적인 분류로 사용할 수 있습니다.


In [ ]:
category_counts = df['category'].value_counts().rename_axis('category').reset_index(name='rows')
display(category_counts)


## RAG 사용 가능 여부

현재 전달본은 `rag_ready == True` 행만 최종 통합한 상태입니다. 임베딩에는 `rag_text`를 사용하고, `source_url`, `trust_level`, `category`, `collection_scope`를 메타데이터로 유지하는 것을 권장합니다.


In [ ]:
display(df['rag_ready'].value_counts(dropna=False).rename_axis('rag_ready').reset_index(name='rows'))
display(df[['unified_id','category','collection_scope','title','trust_level','handoff_warning']].head(10))


## 추천 기능 관련 데이터

보스 추천과 장비 성장 추천은 `collection_scope` 값으로 구분해서 사용할 수 있습니다.

- `supplemental_recommendation_rules`: 보스/장비 추천 룰
- `supplemental_recommendation_review_support`: API 요구사항, 답변 가드레일, 추가 보강 백로그

API 실시간 연동 전에는 사용자가 직접 입력한 스펙을 추천 룰과 비교하는 방식으로 구현하는 것이 적합합니다.


In [ ]:
support_scopes = ['supplemental_recommendation_rules', 'supplemental_recommendation_review_support', 'supplemental_korean_story_summary', 'supplemental_class_5th_core_priority', 'supplemental_class_6th_hexa_priority', 'supplemental_market_event_upgrade_timing']
rec = df[df['collection_scope'].isin(support_scopes)]
print(f'recommendation/support/gap rows={len(rec):,}')
display(rec['collection_scope'].value_counts().rename_axis('collection_scope').reset_index(name='rows'))
display(rec['category'].value_counts().rename_axis('category').reset_index(name='rows'))
display(rec[['collection_scope','category','chatbot_purpose','title','trust_level','handoff_warning','rag_text']].head(12))


## 인수인계 결론

다른 담당자에게는 `maple_chatbot_final_dataset.csv`와 `README_HANDOFF.md`를 우선 전달하면 됩니다. 이 데이터셋은 RAG 기반 메이플 정보 챗봇을 만들기에 적합하며, 보스/장비 추천은 현재 기준 데이터 기반 정답형 응답까지 구현할 수 있습니다. 단, 캐릭터명 기반 자동 분석은 추후 NEXON Open API 실시간 연동이 필요합니다.

추가 보강 데이터도 최종 CSV 안에 실제 seed 행으로 반영했습니다. `supplemental_korean_story_summary`, `supplemental_class_5th_core_priority`, `supplemental_class_6th_hexa_priority`, `supplemental_market_event_upgrade_timing` 범위를 확인하면 됩니다.
